In [1]:
%pip install scapy
import scapy
from scapy.all import *
from socket import *

ip_destino = "8.8.8.8" #IP de google por ej.

puerto_destino = 6789 # Algún puerto que nos guste

puerto_origen = 5000 # Otro puerto que nos guste

# Creamos el paquete
udp_packet = IP(dst=ip_destino)/UDP(dport=puerto_destino, sport=puerto_origen)/Raw(load="nuestra info") #Armemos nuestro paquete. Para agregarle payload, le agregamos /RAW(load="nuestra info")

udp_packet.show() # Inspeccionamos el paquete antes de mandarlo

# Mandamos el paquete
send(udp_packet)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\matia\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 22.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.
###[ IP ]###
  version   = 4
  ihl       = None
  tos       = 0x0
  len       = None
  id        = 1
  flags     = 
  frag      = 0
  ttl       = 64
  proto     = udp
  chksum    = None
  src       = 10.9.1.187
  dst       = 8.8.8.8
  \options   \
###[ UDP ]###
     sport     = 5000
     dport     = 6789
     len       = None
     chksum    = None
###[ Raw ]###
        load      = b'nuestra info'


Sent 1 packets.


In [ ]:
from socket import *
import sys

serverSocket = socket(AF_INET, SOCK_STREAM)

# Preparar el socket del server
serverSocket.bind(('', 6789))  # Bind a puerto 6789 en todas las interfaces
serverSocket.listen(1)  # Permitir 1 conexión simultánea

while True:
    print('Ready to serve...')
    connectionSocket, addr = serverSocket.accept()  # Aceptar la conexión
    try:
        message = connectionSocket.recv(1024).decode()  # Recibir el mensaje
        print(f"Request received: {message.split()[0]} {message.split()[1]}")
        filename = message.split()[1]  # Resolver qué recurso pide el mensaje (segunda palabra del request HTTP)
        f = open(filename[1:], 'rb')  # Abrir archivo en modo binario
        data_respuesta = f.read()  # Cargar el recurso completo
        f.close()

        # Enviar el header HTTP de la respuesta
        connectionSocket.send("HTTP/1.1 200 OK\r\n".encode())
        connectionSocket.send("Content-Type: text/html\r\n".encode())
        connectionSocket.send("\r\n".encode())  # Línea vacía que separa header del body

        # Las siguientes líneas envian el contenido de la respuesta
        connectionSocket.send(data_respuesta)
        connectionSocket.close()

    except IOError:
        # En caso de no tener el archivo, enviar mensaje HTTP correspondiente
        connectionSocket.send("HTTP/1.1 404 Not Found\r\n".encode())
        connectionSocket.send("Content-Type: text/html\r\n".encode())
        connectionSocket.send("\r\n".encode())
        connectionSocket.send("<html><body><h1>404 Not Found</h1></body></html>\r\n".encode())
        connectionSocket.close()

serverSocket.close()
sys.exit()

Ready to serve...
